In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Load your cleaned master dataset
df = pd.read_csv('/content/oulad_cleaned_master.csv')

In [ ]:
# Define features (exclude non-predictive columns)
exclude_cols = ['id_student', 'code_module', 'code_presentation',
                'final_result', 'risk_level', 'is_dropout']
feature_cols = [col for col in df.columns if col not in exclude_cols]
X = df[feature_cols].copy()  # Add .copy() here
y = df['is_dropout'].copy()

# Encode categorical columns
categorical_cols = X.select_dtypes(include=['object']).columns
for col in categorical_cols:
    le = LabelEncoder()
    X.loc[:, col] = le.fit_transform(X[col].astype(str))  # Use .loc

# Handle missing values
for col in X.columns:
    if X[col].dtype == 'object':
        X.loc[:, col] = X[col].fillna(X[col].mode()[0])
    else:
        X.loc[:, col] = X[col].fillna(X[col].median())

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

/tmp/ipykernel_3784/3685385022.py:17: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X.loc[:, col] = X[col].fillna(X[col].mode()[0])


In [ ]:
# Split data (e.g., 80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(random_state=42),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss', use_label_encoder=False)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    results[name] = {'model': model, 'accuracy': accuracy, 'predictions': y_pred}
    print(f'{name} Accuracy: {accuracy:.4f}')

Logistic Regression Accuracy: 0.8199
Random Forest Accuracy: 0.8406


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [05:06:22] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost Accuracy: 0.8432


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

for name, data in results.items():
    print(f'\n=== {name} ===')
    print('Confusion Matrix:')
    print(confusion_matrix(y_test, data['predictions']))
    print('\nClassification Report:')
    print(classification_report(y_test, data['predictions']))

    # ROC-AUC score
    if hasattr(data['model'], 'predict_proba'):
        y_proba = data['model'].predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, y_proba)
        print(f'ROC-AUC Score: {auc:.4f}')


=== Logistic Regression ===
Confusion Matrix:
[[3935  553]
 [ 621 1410]]

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.88      0.87      4488
           1       0.72      0.69      0.71      2031

    accuracy                           0.82      6519
   macro avg       0.79      0.79      0.79      6519
weighted avg       0.82      0.82      0.82      6519

ROC-AUC Score: 0.8892

=== Random Forest ===
Confusion Matrix:
[[4021  467]
 [ 572 1459]]

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.90      0.89      4488
           1       0.76      0.72      0.74      2031

    accuracy                           0.84      6519
   macro avg       0.82      0.81      0.81      6519
weighted avg       0.84      0.84      0.84      6519

ROC-AUC Score: 0.9146

=== XGBoost ===
Confusion Matrix:
[[4005  483]
 [ 539 1492]]

Classification Report:
              precision   

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0]
}

xgb = XGBClassifier(random_state=42, eval_metric='logloss')
grid = GridSearchCV(xgb, param_grid, cv=5, scoring='roc_auc', n_jobs=-1)
grid.fit(X_train, y_train)

print(f"Best params: {grid.best_params_}")
print(f"Best CV ROC-AUC: {grid.best_score_:.4f}")

best_xgb = grid.best_estimator_
y_pred_tuned = best_xgb.predict(X_test)
print(f"Tuned test accuracy: {accuracy_score(y_test, y_pred_tuned):.4f}")

Best params: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200, 'subsample': 1.0}
Best CV ROC-AUC: 0.9202
Tuned test accuracy: 0.8472


In [ ]:
importances = best_xgb.feature_importances_
feature_importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': importances
}).sort_values('importance', ascending=False)

print(feature_importance_df.head(10))

                 feature  importance
11     total_assessments    0.365137
12      late_submissions    0.129620
9            active_days    0.120651
0                 gender    0.075588
8           total_clicks    0.073039
5   num_of_prev_attempts    0.053202
6        studied_credits    0.052842
10             avg_score    0.033159
7             disability    0.022772
1                 region    0.022633
